# IFT 6758 - Homework 3

### Pt 2. Chess Ratings
**Comparing over/underrepresented groups**

Grading for this notebook:

- Plots for the rating distributions and differences
- Printed results of the permutation tests
- Short answers (last section)


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

In [ ]:
import chess_rating

## 1. Load the data

### 1.1 Read the XML data

In [ ]:
xml_path = Path("data/standard_oct22frl_xml.xml")  # shouldn't need to change this

In [ ]:
# TODO: Complete this method in chess_ratings.py
raw_df = chess_rating.parse_xml(xml_path)

In [ ]:
# Check a sample of 5 players
np.random.seed(0)
raw_df.sample(5)

In [ ]:
# Check the dtypes of the raw dataframe
raw_df.dtypes

In [ ]:
# See how many birthdays are missing
raw_df.birthday.isnull().sum()

### 1.2 Clean the data

We want the birthdays (birth year) and the rating (Elo) to be converted to integers so that we can do numerical/logical operations with them (e.g. less than, greater than, etc.). Complete the `clean_data()` method in `chess_rating.py`

In [ ]:
# TODO: Complete this method in chess_ratings.py
df = chess_rating.clean_data(raw_df, 2002)

In [ ]:
np.random.seed(0)
df.sample(5)

In [ ]:
df.dtypes

In [ ]:
df.birthday.isnull().sum()

## 2. Histograms of Elo ratings

We first want to compare the distribution of ratings for male and female players.
Since the data is quite fine-grained, we will need to bin the ratings.
Complete `bin_counts()`, which should handle the binning for arbitrary data and choice of bins.
In addition to returning the raw counts, also return the normalized counts in the dataframe (`"count_norm"`).

### 2.1 Binning data

Here are some test inputs to help you validate your implementation:

In [ ]:
# sample inputs to help test your method
x = pd.DataFrame({'rating': [1, 2, 4, 6, 6, 7, 8, 11] })
bins = [0, 5, 10, 15]
bin_centers = [2.5, 7.5, 12.5]

# Expected output
solution = pd.DataFrame({
    'rating': {0: 7.5, 1: 2.5, 2: 12.5},
     'count': {0: 4, 1: 3, 2: 1},
     'count_norm': {0: 0.5, 1: 0.375, 2: 0.125}
})

In [ ]:
binned_output = chess_rating.bin_counts(x, bins, bin_centers)
binned_output

In [ ]:
solution.eq(binned_output)

Let's use this method to bin the M/F ratings and compare them.

In [ ]:
# TODO: Specify the bin counts
bins = None
bin_centers = None

In [ ]:
# Generate the histogram data
# TODO: complete chess_rating.bin_counts()
M_hist = chess_rating.bin_counts(df.query("sex=='M'"), bins=bins, bin_centers=bin_centers)
F_hist = chess_rating.bin_counts(df.query("sex=='F'"), bins=bins, bin_centers=bin_centers)

M_hist['sex'] = 'M'
F_hist['sex'] = 'F'

hist = pd.concat([M_hist, F_hist])

Using the binned data, draw two line plots of the binned data side by side; one containing the raw counts (`"count"`), and the other containing the normalized counts (`"count_norm"`), and M/F should be two different colors.

In [ ]:
# Now draw the two histograms side-by-side.
fig, ax = plt.subplots(1,2, figsize=(12, 4), dpi=120)
sns.lineplot(data=hist, x="rating", y="count", hue='sex', ax=ax[0])
sns.lineplot(data=hist, x="rating", y="count_norm", hue='sex', ax=ax[1])

## 3. Permutation Tests

We will now perform the permutation tests as described in the thought experiment in the introduction.
Take the full cleaned dataset (men and women) and randomly sample two groups without replacement (i.e. shuffle the players).
The size of the groups should reflect the real-world difference that we wish to study, i.e. the size of the male and female groups.
Finish `chess_rating.PermutationTests.job()`, which implements the sampling part of this experiment, and returns the maximum value of the overrepresented and underrepresented groups respectively.

Then, complete the `chess_rating.sample_two_groups()` method, which runs this experiment `n_iter` times.
Once finished, run this experiment in the notebook with at least `n_iter=1000`.
Run the cell that prints the mean difference obtained from the permutation tests, as well as the real differences.

In [ ]:
n_overrep = None  # TODO
n_underrep = None  # TODO

over, under = chess_rating.sample_two_groups(df, n_overrep, n_underrep, n_iter=1000)

In [ ]:
diffs = over - under
max_M, max_F = df.query("sex=='M'").rating.max(), df.query("sex=='F'").rating.max()

In [ ]:
print(f"Mean difference between top individuals in the over/under represented groups taken from the permutation experiments:")
print(f"\n\t{diffs.mean()} (std={diffs.std().round(2)})")
print()
print("Real values:\n")
print(f"\tMax M:\t\t{max_M}")
print(f"\tMax F:\t\t{max_F}")
print(f"\tReal difference: {max_M - max_F}")

In [ ]:
fig, ax = plt.subplots(1, 1, dpi=100)
sns.histplot(diffs, ax=ax).set(title="Histogram of differences", xlabel="max(overrep) - max(underrep)");

# Questions

Give short answers to the following questions (1-3 lines each).

1. Interpret the results - can you draw a conclusion? Remember that the claim discussed in the introduction to this question was "men are better than women at chess because most of the best players are men". (Note: part of your answer here will probably be related to your answer to the next question.)

*Your answer*


2. Do you think the numbers obtained here tell the whole story? What could be the problems with the analysis conducted here? Is the data we are working with biased in any way (other than an overrepresentation bias)? Is Elo a good measure, and can it be used to answer the original question? Are there differences in the social, cultural and systemic treatment of men and women that may prevent the underrepresented group from achieving similar results? Anything else?

*Your answer*


The goal of these questions is to emphasize that data is a limited representation of the real world. It is essential for us, as data scientists, to take a step back when looking at a result and to think about how it relates to the real world, rather than naively assuming that the data and the experimental setup are good, which often leads to erroneous/incorrect conclusions. There could be several causal factors that explain a relationship that are independent of the original hypothesis: use of data that does not truly reflect the hypothesis you wish to test, biased data (including overrepresented groups), real systemic differences between the groups, etc.